<a href="https://colab.research.google.com/github/ashsiendo/DATA-201-Labs/blob/main/Assignments/Week8_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Week 8 Homework** - Ash Ibasan
We'll use the `Hitters` dataset from the lab in week 8 to see how changing the regularization strength (`α`) affects Lasso regression models, and look at training and test RMSE, how many predictors are kept, and how easy the model is to interpret.

The objective is to compare the results and consider when it makes sense to use ordinary linear regression, Ridge, or Lasso in different modeling scenarios.

> **Note to self: Comments throughout are used as guides, mainly for Python syntax, reasons why certain steps are used, and understanding and interpreting the results.**

## Setup Data Preparation and Preprocessing

*We're going to be predicting professional baseball players' salaries using Lasso regression, and we'll utilize the following: prep the `Hitters` dataset, training/test split, and preprocessing steps*

In [ ]:
%pip -q install ISLP

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# import relevant libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ISLP import load_data
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error

In [ ]:
# load Hitters dataset
hitters = load_data('Hitters')
hitters = hitters.dropna(subset=['Salary']).copy() # remove rows with missing salary values

# define predictors (X); response variable (y)
X = hitters.drop(columns='Salary')
y = hitters['Salary']  # salary in thousands of dollars

# identify numerical and categorical predictors
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split( # training/test split
    X, y, test_size=0.25, random_state=42
)

print('Total observations:', len(X))
print('Training observations:', len(X_train))
print('Test observations:', len(X_test))
print('Categorical predictors:', cat_cols)

Total observations: 263
Training observations: 197
Test observations: 66
Categorical predictors: ['League', 'Division', 'NewLeague']


***Note to self:*** `Salary` is a continuous numeric response, so the problem is regression rather than classification. We'll be using RMSE to measure our prediction error rather than some classification metrics e.g., accuracy, precision, or recall

- `dropna(subset=['Salary'])` removes rows where the `Salary` value (response variable) is missing. Using `.copy()` makes a new DataFrame with the remaining data.

- `X` contains the predictor variables, and `y` is the response variable (`Salary`). We take `Salary` out of `X`, so we don't use the outcome as a predictor.

- `select_dtypes()` splits numerical and categorical predictors so we can preprocess them in different ways.

- `test_size=0.25` sets aside 25% of the data for testing, and `random_state=42` makes sure the split is the same each time we run the code.


We train the models using the training data. The test data then helps us see how well the models predict salaries for new cases.

In [ ]:
def make_preprocessor(): # preprocessing function
    return ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(
                handle_unknown='ignore',
                drop='first'
            ), cat_cols)
        ]
    )

**Note to self:** Lasso penalizes coefficient sizes, so it's important to standardize numerical predictors. This way, the effect of differences in numerical predictor scales on regularization

- `StandardScaler()` is used to standardize numerical predictors
- `OneHotEncoder()` turns categorical variables into numerical indicator columns, so no artificial order is assigned
- Setting `drop='first'` removes one category from each categorical predictor to avoid redundant indicator columns
- The encoder can process categories in the test data that did not appear during training with `handle_unknown='ignore'`
- `ColumnTransformer` applies the right preprocessing steps to each group of predictors

Putting this preprocessor in a `Pipeline` makes sure that scaling and encoding are fitted only on the training data (not test data)

